In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch02-smarts-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "pillow==13.0.0"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "02-03.py"

## 子結構搜尋與 SMARTS

先用五個概念對照核對官能基，再搜尋完整 ESOL 的 1,128 筆記錄。圖中前五格為概念對照，最後一格由 ESOL 篩出。首格準備固定版本環境，末格執行分析並顯圖。第一次執行需要網路下載資料。


In [ ]:
%%writefile "{script}"
# 概念示意 — 實際應用請使用下方 DeepChem 版本
from pathlib import Path
import os
import tempfile
from rdkit import Chem, rdBase
from rdkit.Chem import Draw
import deepchem as dc

patterns = {
    "O": "O",
    "OH": "[OX2H1+0]",
    "alcohol": "[OX2H1+0]-[CX4]",
    "phenol": "[OX2H1+0]-c",
    "carboxylic acid": "[OX2H1+0]-[CX3](=O)",
}
queries = {name: Chem.MolFromSmarts(s) for name, s in patterns.items()}
assert all(q is not None for q in queries.values())
controls = {
    "ethanol": "CCO", "phenol": "Oc1ccccc1", "acetic acid": "CC(=O)O",
    "anisole": "COc1ccccc1", "phenoxide": "[O-]c1ccccc1",
}
expected = [[1,1,1,0,0], [1,1,0,1,0], [1,1,0,0,1], [1,0,0,0,0], [1,0,0,0,0]]
print("Control |", " | ".join(queries))
for (name, smiles), want in zip(controls.items(), expected):
    mol = Chem.MolFromSmiles(smiles)
    got = [int(mol.HasSubstructMatch(q)) for q in queries.values()]
    assert got == want, (name, got)
    print(name, got)

# ESOL：1,128 筆；目標為 log10(S / mol L^-1)。原始論文 DOI: 10.1021/ci034243x。
# 真實資料由 DeepChem 下載；不覆寫 CSV，不以手寫結構代替資料。
cache = Path(os.environ.get("DEEPCHEM_DATA_DIR", Path(tempfile.gettempdir()) / "book-moleculenet"))
cache.mkdir(parents=True, exist_ok=True)
tasks, datasets, transformers = dc.molnet.load_delaney(
    featurizer="Raw", splitter="scaffold", transformers=[],
    data_dir=str(cache), save_dir=str(cache), reload=False,
)
# 不做模型評估，合併三個 split 只為搜尋全資料；標的未做 normalization。
smiles = [str(s) for part in datasets for s in part.ids]
mols = [Chem.MolFromSmiles(s) for s in smiles]
assert len(mols) == 1128 and all(m is not None for m in mols)
assert transformers == []
# 排序及畫圖均為確定性操作，不使用隨機抽樣。
print("RDKit", rdBase.rdkitVersion, "DeepChem", dc.__version__)
print("ESOL rows:", len(mols), "normalization:", bool(transformers))
for name, query in queries.items():
    count = sum(mol.HasSubstructMatch(query) for mol in mols)
    print(name, "matched rows:", count)

query = queries["alcohol"]
hits = [(i, mol.GetSubstructMatches(query)) for i, mol in enumerate(mols)
        if mol.HasSubstructMatch(query)]
print("alcohol rows:", len(hits))
print("alcohol sites:", sum(len(matches) for _, matches in hits))

# 五個概念對照加一筆 ESOL 的多羥基分子；圖例區分來源。
images = [Chem.MolFromSmiles(s) for s in controls.values()]
labels = list(controls)
query_names = ["alcohol", "phenol", "carboxylic acid", "OH", "OH"]
# 從真實 ESOL 找到最小的雙醇位點分子，順序完全固定。
indices = [i for i, m in enumerate(mols) if len(m.GetSubstructMatches(queries["alcohol"])) == 2]
indices.sort(key=lambda i: (mols[i].GetNumHeavyAtoms(), Chem.MolToSmiles(mols[i]), i))
i = indices[0]
images.append(mols[i])
labels.append("ESOL: " + Chem.MolToSmiles(mols[i]))
query_names.append("alcohol")
legends, atom_lists, bond_lists = [], [], []
for j, (mol, label, name) in enumerate(zip(images, labels, query_names)):
    q = queries[name]
    matches = mol.GetSubstructMatches(q)
    atoms = sorted({atom for match in matches for atom in match})
    bonds = sorted({mol.GetBondBetweenAtoms(match[b.GetBeginAtomIdx()], match[b.GetEndAtomIdx()]).GetIdx()
                    for match in matches for b in q.GetBonds()})
    source = "Concept: " if j < 5 else ""
    legends.append(f"{source}{label}\n{name} sites: {len(matches)}")
    atom_lists.append(atoms)
    bond_lists.append(bonds)
    print("draw:", label, name, matches)
fig = Draw.MolsToGridImage(images, molsPerRow=3, subImgSize=(340, 240),
                          legends=legends, highlightAtomLists=atom_lists,
                          highlightBondLists=bond_lists)
out = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
out.mkdir(exist_ok=True)
fig.save(out / "02-03-substructure-smarts.png")


In [ ]:
from IPython.display import Image, display
run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.strip())
figure_dir = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "02-03-substructure-smarts.png")))
